In [6]:
from pathlib import Path
import numpy as np
import struct
import os.path
import operator
import copy

itemids = [220051, 220052, 220210, 224695, 224687, 224697, 220074, 224688, 220061, 223834, 220047, 220059,
           220060, 224161, 220045, 223761, 1529, 226707, 220050, 224639]

'''
data_folder1 = Path("C:/Users/IICT/JOY/Patient Similarity/SimCodeData/Minhash/heartpatients")
data_folder2 = Path("C:/Users/IICT/JOY/Patient Similarity/SimCodeData/Clustering/Sorted_Series")
data_folder3 = Path("C:/Users/IICT/JOY/Patient Similarity/SimCodeData/Clustering")
data_folder4 = Path("C:/Users/IICT/JOY/Patient Similarity/SimCodeData/dtw/Series/Distance_Matrices")
output_folder = Path("C:/Users/IICT/JOY/Patient Similarity/SimCodeData/Clustering/Output")
'''

data_folder1 = Path("C:/Users/Joy/Python-CODE/patient-similarity-code/Datasets/SimCodeData/Minhash/heartpatients")
data_folder2 = Path("C:/Users/Joy/Python-CODE/patient-similarity-code/Datasets/SimCodeData/Clustering/Sorted_Series")
data_folder3 = Path("C:/Users/Joy/Python-CODE/patient-similarity-code/Datasets/SimCodeData/Clustering")
data_folder4 = Path("C:/Users/Joy/Python-CODE/patient-similarity-code/Datasets/SimCodeData/dtw/Series/Distance_Matrices")
output_folder = Path("C:/Users/Joy/Python-CODE/patient-similarity-code/Datasets/SimCodeData/Clustering/Output")

size = 20
All_Series_Confusion_matrix = np.zeros((size, 21))
All_Series_Accuracy = np.zeros((size, 3))

available_patients = np.loadtxt(os.path.join(data_folder1, 'available_patients.csv'), delimiter=',', dtype=int)
all_patients = np.load(os.path.join(data_folder1, 'heartpatients_sortedby_subject_id_with_duration.npy'))
all_patients = np.delete(all_patients, 0, 0)
adm_types = np.loadtxt(os.path.join(data_folder1, 'heartpatients_sortedby_subject_id.csv'), delimiter='\t', dtype=str)
adm_ids = all_patients[:, 1]
adm_ids = list(adm_ids)
adm_ids = [int(x) for x in adm_ids]
adm_types_adm_ids = adm_types[:, 0]
adm_types_adm_ids = list(adm_types_adm_ids)
adm_types_adm_ids = [int(x) for x in adm_types_adm_ids]

similarity_Matrix = list(np.loadtxt(os.path.join(data_folder3, 'Real_Patient_Index' + '.csv'), delimiter=',',dtype=float))
real_Index = similarity_Matrix
real_Index = list(real_Index)
real_Index = [int(x) for x in real_Index]

class process_Count_Series:

    def calculate_Accuracy(self, patient_Relation, voting_Matrix):
        expiry_Correct = 0
        expiry_Wrong = 0
        diagnosis_Correct = 0
        diagnosis_Wrong = 0
        emergency_Correct = 0
        emergency_Wrong = 0
        #series_Index = index
        #All_Series_Confusion_matrix = np.zeros((size, 21))

        for i in range(patient_Relation.__len__()):
            
            index1 = (int(patient_Relation[i][0]))
            index2 = (int(patient_Relation[i][1]))

            id1 = int(real_Index[index1])
            id2 = int(real_Index[index2])

            index1 = adm_ids.index(id1)
            index2 = adm_ids.index(id2)
            r1 = all_patients[index1]
            r2 = all_patients[index2]
            
            """
            if int(r1[4]) == int(r2[4]):
                if int(r1[4]) == 0:
                    All_Series_Confusion_matrix[series_Index][0] += 1
                else:
                    All_Series_Confusion_matrix[series_Index][3] += 1
                expiry_Correct += 1
            else:
                if int(r1[4]) == 0:
                    All_Series_Confusion_matrix[series_Index][1] += 1
                else:
                    All_Series_Confusion_matrix[series_Index][2] += 1
                expiry_Wrong += 1
               
               """
            cong_hear_failure1 = str(r1[2])
            cong_hear_failure2 = str(r2[2])
            

            cong_hear_failure1 = cong_hear_failure1.replace('//', ',')
            cong_hear_failure1 = cong_hear_failure1.replace('\\', ',')
            cong_hear_failure1 = cong_hear_failure1.replace("\\", ",")
            cong_hear_failure1 = cong_hear_failure1.replace('/', ',')
            cong_hear_failure1 = cong_hear_failure1.replace(';', ',')
            cong_hear_failure1 = cong_hear_failure1.replace(':', ',')
            cong_hear_failure1 = cong_hear_failure1.split(',')
            for i1 in range(cong_hear_failure1.__len__()):
                cong_hear_failure1[i1] = str(cong_hear_failure1[i1]).strip()

            cong_hear_failure2 = cong_hear_failure2.replace('//', ',')
            cong_hear_failure2 = cong_hear_failure2.replace('\\', ',')
            cong_hear_failure2 = cong_hear_failure2.replace("\\", ",")
            cong_hear_failure2 = cong_hear_failure2.replace('/', ',')
            cong_hear_failure2 = cong_hear_failure2.replace(';', ',')
            cong_hear_failure2 = cong_hear_failure2.replace(':', ',')
            cong_hear_failure2 = cong_hear_failure2.split(',')
            for i2 in range(cong_hear_failure2.__len__()):
                cong_hear_failure2[i2] = str(cong_hear_failure2[i2]).strip()
                
            frequenct_cong_hear_failure = 'CONGESTIVE HEART FAILURE'
 
            if frequenct_cong_hear_failure in cong_hear_failure1 and frequenct_cong_hear_failure in cong_hear_failure2:
                voting_Matrix[i][3] += 1 #tp 12
               
            elif frequenct_cong_hear_failure not in cong_hear_failure1 and frequenct_cong_hear_failure not in cong_hear_failure2:
                voting_Matrix[i][0] += 1 #tn  15
                
            else:
                if frequenct_cong_hear_failure in cong_hear_failure2:
                    voting_Matrix[i][1] += 1 #fp 13
                else:
                    voting_Matrix[i][2] += 1 #fn  14
            
                        
        
            diagnosis1 = str(r1[2])
            diagnosis2 = str(r2[2])

            diagnosis1 = diagnosis1.replace('//', ',')
            diagnosis1 = diagnosis1.replace('\\', ',')
            diagnosis1 = diagnosis1.replace("\\", ",")
            diagnosis1 = diagnosis1.replace('/', ',')
            diagnosis1 = diagnosis1.replace(';', ',')
            diagnosis1 = diagnosis1.replace(':', ',')
            diagnosis1 = diagnosis1.split(',')
            for i1 in range(diagnosis1.__len__()):
                diagnosis1[i1] = str(diagnosis1[i1]).strip()

            diagnosis2 = diagnosis2.replace('//', ',')
            diagnosis2 = diagnosis2.replace('\\', ',')
            diagnosis2 = diagnosis2.replace("\\", ",")
            diagnosis2 = diagnosis2.replace('/', ',')
            diagnosis2 = diagnosis2.replace(';', ',')
            diagnosis2 = diagnosis2.replace(':', ',')
            diagnosis2 = diagnosis2.split(',')
            for i2 in range(diagnosis2.__len__()):
                diagnosis2[i2] = str(diagnosis2[i2]).strip()

            most_frequenct_diagnosis = 'CORONARY ARTERY DISEASE'

            if most_frequenct_diagnosis in diagnosis1 and most_frequenct_diagnosis in diagnosis2:
                voting_Matrix[i][7] += 1 #tp
                diagnosis_Correct += 1
            elif most_frequenct_diagnosis not in diagnosis1 and most_frequenct_diagnosis not in diagnosis2:
                voting_Matrix[i][4] += 1 #tn
                diagnosis_Correct += 1
            else:
                if most_frequenct_diagnosis in diagnosis2:
                    voting_Matrix[i][5] += 1 #fp
                else:
                    voting_Matrix[i][6] += 1 #fn
                diagnosis_Wrong += 1


            index1 = adm_types_adm_ids.index(id1)
            index2 = adm_types_adm_ids.index(id2)
            r1 = adm_types[index1]
            r2 = adm_types[index2]

   
         
        
            if str(r1[4]).strip() == 'EMERGENCY' and str(r2[4]).strip() == 'EMERGENCY':
                voting_Matrix[i][11] += 1 #tp
                emergency_Correct += 1
            elif str(r1[4]).strip() != 'EMERGENCY' and str(r2[4]).strip() != 'EMERGENCY':
                voting_Matrix[i][8] += 1 #tn
                emergency_Correct += 1
            else:
                if str(r2[4]).strip() == 'EMERGENCY':
                    voting_Matrix[i][9] += 1 #fp
                elif str(r1[4]).strip() == 'EMERGENCY':
                    voting_Matrix[i][10] += 1 #fn
                emergency_Wrong += 1      

        #print("-----------------------------------------------------")
        #print("Expiray flag accuracy is " + str(expiry_Percentage) + " %")
        #print("Diagnosis flag accuracy is " + str(diagnosis_Percentage) + " %")
        #print("Emergency flag accuracy is " + str(emergency_Percentage) + " %")
        
        
        
    def evaluate_Model(self, V_clustering_Method, voting_Matrix):
        tn=0 
        fp=0
        fn=0
        tp=0
        
        for i in range(voting_Matrix.__len__()):
            
            # voting_Matrix[i][0] ---> tn
            # voting_Matrix[i][1] ---> fp
            # voting_Matrix[i][2] ---> fn
            # voting_Matrix[i][3] ---> tp
            maxValue=max(voting_Matrix[i][0], voting_Matrix[i][1], voting_Matrix[i][2], voting_Matrix[i][3])
            if maxValue == voting_Matrix[i][3]:
                tp+=1
            elif maxValue == voting_Matrix[i][0]:
                tn+= 1
            elif maxValue == voting_Matrix[i][1]:
                fp+=1
            else:
                fn+=1
             
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        f1 = 2 * (prec * rec) / (prec + rec)
        if V_clustering_Method=='CONG_HEART_FAILURE':
            print("CONG_HEART_FAILURE flag F1 is " + str(f1) + " %")
            print("tn, fp, fn, tp " +   str(tn) +", " + str(fp)+", " + str(fn)+", "+ str(tp) )
            pod=tp/(tp+fn)
            pof=fp/(fp+tn)
            auc_val=(1+pod-pof)/2
            accuracy_val=(tp+tn)/(tp+fn+fp+tn)
            precision_val=tp/(tp+fp)  
            recall_val = tp / (tp + fn)
            specificity_val=tn/(fp+tn)
            F1_val=(2*precision_val*recall_val)/(precision_val+recall_val)
            print ('AUC: ',auc_val)
            print ('accuracy: ',accuracy_val)
            print ('F1: ',F1_val)
            print ('precision: ',precision_val)
            print ('recall: ',recall_val)
            print ('specificity: ',specificity_val)
            print ('F1: ',F1_val)
            

        tn=0 
        fp=0
        fn=0
        tp=0
        
        for i in range(voting_Matrix.__len__()):
            
            # voting_Matrix[i][4] ---> tn
            # voting_Matrix[i][5] ---> fp
            # voting_Matrix[i][6] ---> fn
            # voting_Matrix[i][7] ---> tp
            maxValue=max(voting_Matrix[i][4], voting_Matrix[i][5], voting_Matrix[i][6], voting_Matrix[i][7])
            if maxValue == voting_Matrix[i][7]:
                tp+=1
            elif maxValue == voting_Matrix[i][4]:
                tn+= 1
            elif maxValue == voting_Matrix[i][5]:
                fp+=1
            else:
                fn+=1
 
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        f1 = 2 * (prec * rec) / (prec + rec)
        if V_clustering_Method=='Diagnosis':
            print("DIAGNOSIS flag F1 is " + str(f1) + " %")
            print("tn, fp, fn, tp" +   str(tn) +", " + str(fp)+", " + str(fn)+", "+ str(tp) )
            pod=tp/(tp+fn)
            pof=fp/(fp+tn)
            auc_val=(1+pod-pof)/2
            accuracy_val=(tp+tn)/(tp+fn+fp+tn)
            precision_val=tp/(tp+fp)  
            recall_val = tp / (tp + fn)
            specificity_val=tn/(fp+tn)
            F1_val=(2*precision_val*recall_val)/(precision_val+recall_val)
            print ('AUC: ',auc_val)
            print ('accuracy: ',accuracy_val)
            print ('F1: ',F1_val)
            print ('precision: ',precision_val)
            print ('recall: ',recall_val)
            print ('specificity: ',specificity_val)
            print ('F1: ',F1_val)
        
        tn=0 
        fp=0
        fn=0
        tp=0
        
        for i in range(voting_Matrix.__len__()):
            
            # voting_Matrix[i][8] ---> tn
            # voting_Matrix[i][9] ---> fp
            # voting_Matrix[i][10] ---> fn
            # voting_Matrix[i][11] ---> tp
            maxValue=max(voting_Matrix[i][8], voting_Matrix[i][9], voting_Matrix[i][10], voting_Matrix[i][11])
            if maxValue == voting_Matrix[i][11]:
                tp+=1
            elif maxValue == voting_Matrix[i][8]:
                tn+= 1
            elif maxValue == voting_Matrix[i][9]:
                fp+=1
            else:
                fn+=1
 
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        f1 = 2 * (prec * rec) / (prec + rec)
        if V_clustering_Method=='Emergency':
            print("EMERGENCY ADMISSION flag F1 is " + str(f1) + " %")
            print("tn, fp, fn, tp" +   str(tn) +", " + str(fp)+", " + str(fn)+", "+ str(tp) )
            pod=tp/(tp+fn)
            pof=fp/(fp+tn)
            auc_val=(1+pod-pof)/2
            accuracy_val=(tp+tn)/(tp+fn+fp+tn)
            precision_val=tp/(tp+fp)  
            recall_val = tp / (tp + fn)
            specificity_val=tn/(fp+tn)
            F1_val=(2*precision_val*recall_val)/(precision_val+recall_val)
            print ('AUC: ',auc_val)
            print ('accuracy: ',accuracy_val)
            print ('F1: ',F1_val)
            print ('precision: ',precision_val)
            print ('recall: ',recall_val)
            print ('specificity: ',specificity_val)
            print ('F1: ',F1_val)
             
             
        """"          
        expiry_Correct = All_Series_Confusion_matrix[series_Index][0] + All_Series_Confusion_matrix[series_Index][3]
        expiry_Wrong = 4418*19 - expiry_Correct
        diagnosis_Correct = All_Series_Confusion_matrix[series_Index][4] + All_Series_Confusion_matrix[series_Index][7]
        diagnosis_Wrong = 4418*19 - diagnosis_Correct
        print("diagnosis_Correct:" +str(diagnosis_Correct))
        emergency_Correct = All_Series_Confusion_matrix[series_Index][8] + All_Series_Confusion_matrix[series_Index][11]
        emergency_Wrong = 4418 - emergency_Correct

        expiry_Percentage = (expiry_Correct / (expiry_Wrong + expiry_Correct)) * 100
        diagnosis_Percentage = (diagnosis_Correct / (diagnosis_Wrong + diagnosis_Correct)) * 100
        emergency_Percentage = (emergency_Correct / (emergency_Wrong + emergency_Correct)) * 100

        All_Series_Accuracy[series_Index][0] = expiry_Percentage
        All_Series_Accuracy[series_Index][1] = diagnosis_Percentage
        All_Series_Accuracy[series_Index][2] = emergency_Percentage

        k = 0
        tp = All_Series_Confusion_matrix[series_Index][0 + k]
        fp = All_Series_Confusion_matrix[series_Index][1 + k]
        fn = All_Series_Confusion_matrix[series_Index][2 + k]
        tn = All_Series_Confusion_matrix[series_Index][3 + k]
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        f1 = 2 * (prec * rec) / (prec + rec)

        All_Series_Confusion_matrix[series_Index][12 + k] = float("{0:.3f}".format(prec))
        All_Series_Confusion_matrix[series_Index][13 + k] = float("{0:.3f}".format(rec))
        All_Series_Confusion_matrix[series_Index][14 + k] = float("{0:.3f}".format(f1))
        print("Expiray flag F1 is " + str(f1) + " %")

        
        k = 4
        tp = All_Series_Confusion_matrix[series_Index][0 + k]
        fp = All_Series_Confusion_matrix[series_Index][1 + k]
        fn = All_Series_Confusion_matrix[series_Index][2 + k]
        tn = All_Series_Confusion_matrix[series_Index][3 + k]
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        f1 = 2 * (prec * rec) / (prec + rec)

        All_Series_Confusion_matrix[series_Index][11 + k] = float("{0:.3f}".format(prec))
        All_Series_Confusion_matrix[series_Index][12 + k] = float("{0:.3f}".format(rec))
        All_Series_Confusion_matrix[series_Index][13 + k] = float("{0:.3f}".format(f1))
        print("Diagnosis flag F1 is " + str(f1) + " %")
            

        k = 8
        tp = All_Series_Confusion_matrix[series_Index][0 + k]
        fp = All_Series_Confusion_matrix[series_Index][1 + k]
        fn = All_Series_Confusion_matrix[series_Index][2 + k]
        tn = All_Series_Confusion_matrix[series_Index][3 + k]
        prec = tp / (tp + fp)
        rec = tp / (tp + fn)
        f1 = 2 * (prec * rec) / (prec + rec)

        All_Series_Confusion_matrix[series_Index][10 + k] = float("{0:.3f}".format(prec))
        All_Series_Confusion_matrix[series_Index][11 + k] = float("{0:.3f}".format(rec))
        All_Series_Confusion_matrix[series_Index][12 + k] = float("{0:.3f}".format(f1))
        print("Emergency flag F1 is " + str(f1) + " %")

        print("-----------------------------------------------------")
        print("Expiray flag accuracy is " + str(expiry_Percentage) + " %")
        print("Diagnosis flag accuracy is " + str(diagnosis_Percentage) + " %")
        print("Emergency flag accuracy is " + str(emergency_Percentage) + " %") 
        """


process_Counts = process_Count_Series()

sorted_Series_Index = np.zeros((4418,4418))
Considered_Row_Index = np.zeros((4418,20))
patient_Counts_All_Series = np.zeros((4418,4418))
not_Missed_Rows = np.zeros((4418,1))

sorted_Series_List = list()

missed_Item = 0
series_Index = 0

Intersection_Set = list()
Considered_Row_Index = list(np.loadtxt(os.path.join(data_folder2, 'Considered_Rows_All_Series' + '.csv'), delimiter=',',dtype=float))
sorted_Series_List.clear()

Methods_list  = ["Diagnosis","Emergency", "CONG_HEART_FAILURE"]
Methods_list  = ["CONG_HEART_FAILURE"]


#Methods_list  = ["CONG_HEART_FAILURE"]
 
#clustering_Method = "Expire"
#clustering_Method = "Diagnosis"
#clustering_Method = "Emergency"
#clustering_Method = "CONG_HEART_FAILURE"

#num_Clusters = 600



#num_Clusters_list=[10, 25, 50, 75, 100, 125, 150, 200, 250, 275, 300, 325, 350, 375, 400, 425, 450, 475, 500, 525, 550, 575, 600]
num_Clusters_list=[375, 400, 425, 450, 475, 500, 525, 550, 575, 600]
#num_Clusters_list=[600]


for num_Clusters in num_Clusters_list:
    #
    print('==================Cluster Number: '+ str(num_Clusters) +"=================")
    for clustering_Method in Methods_list:
        #
        sorted_Series_Index = np.zeros((4418,4418))
        Considered_Row_Index = np.zeros((4418,20))
        patient_Counts_All_Series = np.zeros((4418,4418))
        not_Missed_Rows = np.zeros((4418,1))

        sorted_Series_List = list()

        missed_Item = 0
        series_Index = 0
        #---Z-Score-----------------------
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2024/spectral_Z-Score_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2024/k-mean-Z-Score_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/Agglomerative-Z-Score_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/OPTICS-Z-Score_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))

       # Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/2_' + clustering_Method + '.csv'), delimiter=',',dtype=float))
        
        #---WOE-----------------
        Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2024/spectral_WOE_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2024/k-mean-WOE_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/Agglomerative-WOE_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/OPTICS-WOE_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        
        #---- NO DT method---------------
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/spectral_NO_DT_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'Clustering-OCT-2023/spectral_NO_DT_RAW_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V4.csv'), delimiter=',',dtype=float))
        #Class_IDs = list(np.loadtxt(os.path.join(data_folder3, 'spectral-v2/spectral_Normalized_Static_Data_' + clustering_Method + "_" + str(num_Clusters) + '_Output-V2.csv'), delimiter=',',dtype=float))
 
         

        for item in itemids:
            similarity_Matrix = np.loadtxt(os.path.join(data_folder2, 'Sorted_Series_Indecies_' + str(item) + '.csv'), delimiter=',', dtype=float)
            sorted_Series_List.append(list(similarity_Matrix))

        voting_Matrix = np.zeros((881, 21))
        #print(voting_Matrix)

        series_Index = 0
        for series in sorted_Series_List:
            patient_Relation = np.zeros((881, 2))
            for i in range(series.__len__()):
                patient_ID = Class_IDs[i]
                
                if i==881:
                    break

                """
                if Considered_Row_Index[i][series_Index] == 0:
                    continue
                """

                for j in range(4417):
                    #patient_Index = int(series[i][4417 - j])
                    patient_Index = int(series[i][j])
                    patient_ID_Related = Class_IDs[patient_Index]

                    if patient_ID_Related == patient_ID:
                        patient_Relation[i][0] = i
                        patient_Relation[i][1] = patient_Index
                        break

            #print(itemids[series_Index])
            np.savetxt(os.path.join(output_folder, 'patient_Relation_' + str(itemids[series_Index]) + 'TEST.csv'),patient_Relation, delimiter=',', fmt='%s')
            process_Counts.calculate_Accuracy(patient_Relation,voting_Matrix)
            series_Index += 1



        #np.savetxt(os.path.join(output_folder, 'Accuracy_List_Spectral_Clustering_Model_' + clustering_Method + 'TEST.csv'),All_Series_Accuracy, delimiter=',', fmt='%s')
        #np.savetxt(os.path.join(output_folder, 'Confusion_Matrix_Spectral_Clustering_Model_' + clustering_Method + 'TEST.csv'),All_Series_Confusion_matrix, delimiter=',', fmt='%s')

        #print(voting_Matrix)
        process_Counts.evaluate_Model(clustering_Method, voting_Matrix)


==================Cluster Number: 375=================
CONG_HEART_FAILURE flag F1 is 0.7410358565737052 %
tn, fp, fn, tp 565, 69, 61, 186
AUC:  0.8221018148379928
accuracy:  0.8524404086265607
F1:  0.7410358565737052
precision:  0.7294117647058823
recall:  0.7530364372469636
specificity:  0.8911671924290221
F1:  0.7410358565737052
==================Cluster Number: 400=================
CONG_HEART_FAILURE flag F1 is 0.761904761904762 %
tn, fp, fn, tp 569, 65, 55, 192
AUC:  0.8374021379583392
accuracy:  0.8637911464245176
F1:  0.761904761904762
precision:  0.7470817120622568
recall:  0.7773279352226721
specificity:  0.8974763406940063
F1:  0.761904761904762
==================Cluster Number: 425=================
CONG_HEART_FAILURE flag F1 is 0.7649402390438247 %
tn, fp, fn, tp 571, 63, 55, 192
AUC:  0.8389794250245852
accuracy:  0.8660612939841089
F1:  0.7649402390438247
precision:  0.7529411764705882
recall:  0.7773279352226721
specificity:  0.9006309148264984
F1:  0.7649402390438247
====